<a href="https://colab.research.google.com/github/keathana2007/DEEP-LEARNING/blob/main/Skill3dl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q fastapi uvicorn python-multipart requests


In [2]:

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU Available!")
    print(torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU Not Available")
    print("Using CPU")

print("Selected Device:", device)


GPU Available!
Tesla T4
Selected Device: cuda


In [3]:

import torch
from torchvision import models
from PIL import Image

# Load pretrained model
weights = models.MobileNet_V2_Weights.DEFAULT

model = models.mobilenet_v2(
    weights=weights
)

# Move model to GPU or CPU
model = model.to(device)

# Set model to evaluation mode
model.eval()

# Get image preprocessing method
preprocess = weights.transforms()

print("VisionPort Model Loaded!")
print("Model: MobileNetV2")
print("Device:", device)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 211MB/s]


VisionPort Model Loaded!
Model: MobileNetV2
Device: cuda


In [4]:

def preprocess_image(image):

    # Convert image to RGB
    image = image.convert("RGB")

    # Apply model preprocessing
    image_tensor = preprocess(image)

    # Add batch dimension
    image_tensor = image_tensor.unsqueeze(0)

    # Move image to GPU or CPU
    image_tensor = image_tensor.to(device)

    return image_tensor

print("Image Preprocessing Ready!")


Image Preprocessing Ready!


In [5]:

def run_inference(image_tensor):

    with torch.inference_mode():

        # Execute AI model
        output = model(image_tensor)

        # Calculate probabilities
        probabilities = torch.softmax(
            output[0], dim=0
        )

        # Find top prediction
        confidence, predicted_class = torch.max(
            probabilities, dim=0
        )

        # Get predicted class name
        class_name = weights.meta["categories"][
            predicted_class.item()
        ]

        return {
            "prediction": class_name,
            "confidence": round(
                confidence.item() * 100, 2
            )
        }

print("Inference Function Ready!")


Inference Function Ready!


In [6]:

from fastapi import FastAPI, UploadFile, File, HTTPException
from PIL import Image, UnidentifiedImageError
import io
import time
import uuid

app = FastAPI(
    title="VisionPort Inference Trace",
    description="AI Image Inference Request Tracking",
    version="1.0"
)

@app.get("/")
def home():
    return {
        "project": "VisionPort",
        "status": "Running",
        "message": "Inference API Ready"
    }

@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    request_id = str(uuid.uuid4())
    start_time = time.perf_counter()

    trace = []

    try:
        # STAGE 1: API INPUT
        stage_start = time.perf_counter()

        image_bytes = await file.read()

        trace.append({
            "stage": "API Input",
            "status": "Completed",
            "time_ms": round(
                (time.perf_counter() - stage_start) * 1000, 3
            )
        })

        # STAGE 2: IMAGE DECODING
        stage_start = time.perf_counter()

        image = Image.open(
            io.BytesIO(image_bytes)
        ).convert("RGB")

        image_size = image.size

        trace.append({
            "stage": "Image Decoding",
            "status": "Completed",
            "image_size": image_size,
            "time_ms": round(
                (time.perf_counter() - stage_start) * 1000, 3
            )
        })

        # STAGE 3: PREPROCESSING
        stage_start = time.perf_counter()

        image_tensor = preprocess_image(image)

        trace.append({
            "stage": "Preprocessing",
            "status": "Completed",
            "tensor_shape": list(image_tensor.shape),
            "time_ms": round(
                (time.perf_counter() - stage_start) * 1000, 3
            )
        })

        # STAGE 4: MODEL EXECUTION
        stage_start = time.perf_counter()

        result = run_inference(image_tensor)

        # Wait for GPU operations to finish
        if device.type == "cuda":
            torch.cuda.synchronize()

        trace.append({
            "stage": "Model Execution",
            "status": "Completed",
            "model": "MobileNetV2",
            "device": str(device),
            "time_ms": round(
                (time.perf_counter() - stage_start) * 1000, 3
            )
        })

        # STAGE 5: RESPONSE
        total_time = round(
            (time.perf_counter() - start_time) * 1000, 3
        )

        trace.append({
            "stage": "Response",
            "status": "Completed"
        })

        return {
            "request_id": request_id,
            "filename": file.filename,
            "prediction": result["prediction"],
            "confidence_percent": result["confidence"],
            "image_size": image_size,
            "total_time_ms": total_time,
            "execution_trace": trace
        }

    except (UnidentifiedImageError, OSError, ValueError):
        raise HTTPException(
            status_code=400,
            detail="Invalid or unsupported image"
        )

print("VisionPort Inference Trace API Created!")


VisionPort Inference Trace API Created!


In [7]:

import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def start_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8001,
        log_level="warning"
    )

server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("VisionPort API Started Successfully!")
print("Port: 8001")


VisionPort API Started Successfully!
Port: 8001


In [8]:

from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

print("Traffic Image Uploaded:", filename)


Saving a_wide_clear_daytime_urban_street_road_scene_view (1).png to a_wide_clear_daytime_urban_street_road_scene_view (1).png
Traffic Image Uploaded: a_wide_clear_daytime_urban_street_road_scene_view (1).png


In [9]:

import requests

url = "http://127.0.0.1:8001/predict"

with open(filename, "rb") as image_file:

    response = requests.post(
        url,
        files={
            "file": (
                filename,
                image_file,
                "image/jpeg"
            )
        }
    )

print("HTTP Status:", response.status_code)

data = response.json()

print("\nPREDICTION:")
print(data["prediction"])

print("\nCONFIDENCE:")
print(data["confidence_percent"], "%")

print("\nTOTAL TIME:")
print(data["total_time_ms"], "ms")


HTTP Status: 200

PREDICTION:
trolleybus

CONFIDENCE:
10.97 %

TOTAL TIME:
4006.617 ms


In [10]:

print("VISIONPORT INFERENCE REQUEST TRACE")
print("-----------------------------------")

print("Request ID:", data["request_id"])

for stage in data["execution_trace"]:

    print("\nStage:", stage["stage"])
    print("Status:", stage["status"])

    if "time_ms" in stage:
        print("Time:", stage["time_ms"], "ms")

    if "device" in stage:
        print("Device:", stage["device"])

    if "tensor_shape" in stage:
        print("Tensor Shape:", stage["tensor_shape"])

print("\nTOTAL PROCESSING TIME:",
      data["total_time_ms"], "ms")


VISIONPORT INFERENCE REQUEST TRACE
-----------------------------------
Request ID: 41815f9a-be51-471f-96ee-e848ed860c1a

Stage: API Input
Status: Completed
Time: 2.042 ms

Stage: Image Decoding
Status: Completed
Time: 131.58 ms

Stage: Preprocessing
Status: Completed
Time: 67.958 ms
Tensor Shape: [1, 3, 224, 224]

Stage: Model Execution
Status: Completed
Time: 3805.0 ms
Device: cuda

Stage: Response
Status: Completed

TOTAL PROCESSING TIME: 4006.617 ms
